# 03 · Baseline evaluation — Qwen3-ASR-1.7B on three large Vietnamese corpora

Measures the **out-of-the-box** WER/CER of `Qwen/Qwen3-ASR-1.7B-hf` on the
held-out test split of each of three corpora:

| corpus | source | corpus size | streamed here | character |
|---|---|---|---|---|
| `vivoice_full` | [`capleaf/viVoice`](https://huggingface.co/datasets/capleaf/viVoice) | ~1,000 h | 100 h | YouTube speech, native clips |
| `vietspeech` | [`NhutP/VietSpeech`](https://huggingface.co/datasets/NhutP/VietSpeech) | ~1,100 h | 100 h | social-media speech, 2–6 s clips, all three regional accents |
| `vieneu` | [`pnnbao-ump/VieNeu-TTS-140h`](https://huggingface.co/datasets/pnnbao-ump/VieNeu-TTS-140h) | ~140.7 h | 100 h | studio TTS corpus, 193 voices, clean read speech |

300 h in total — an equal cap per corpus rather than the whole of any of them,
which is both a budget decision (~35 GB, ~4 wall-hours to download) and a
balance decision: the three differ in size by 8x, and taking them in full would
let viVoice and VietSpeech account for 94% of the mixture. Raise
`HOURS_PER_CORPUS` to `None` for the full ~2,240 h; `04_lora_finetune_3ds.ipynb`
must be raised in step.

This is the large-corpus counterpart to `01_eval_baseline.ipynb`, which scores a
single 8 h viVoice slice. `04_lora_finetune_3ds.ipynb` fine-tunes on the same
three corpora and diffs against the numbers written here.

## Which VieNeu release, and why

This uses the **140 h** release, not the 1000 h sibling. The 1000 h repo is
gated `manual` and its authors restrict it to institutions, research labs and
universities; this project's request is still awaiting review, so every file
resolves `403`. The 140 h release is gated `auto` — accepting the terms on the
dataset page is enough — and is otherwise the same corpus shape.

Two things worth knowing about it:

- It ships **both** `text` (ordinary Vietnamese) and `phonemized_text` (IPA).
  `corpora.py` pins the text column to `text`: training on phonemes would
  optimise a target the WER metric never sees.
- Audio is 24 kHz and is resampled to 16 kHz on the way in.

It is also the easiest of the three: clean studio read speech, so expect a much
lower baseline WER here than on viVoice or VietSpeech. That makes it the least
informative of the three for judging real-world gains, and the most likely to
show a small delta simply because there is little headroom.

## How the test splits are built

None of the three ships an official test split, so `corpora.py` builds them —
never at the clip level. Consecutive clips from one recording share a speaker, a
room and a topic, so a random clip-level split leaks the test set into training.
- **viVoice** splits on its real `channel` column.
- **VietSpeech** has no speaker column, so it splits on the recording prefix in
  its filenames (`278_000000086.wav`).
- **VieNeu** has a `speaker` column, but it holds one id per *recording*
  (`jellyfish1010_0041`) across only 193 voices — so the trailing index is
  stripped and the split is by voice, not by recording.

See the `corpora` module docstring for the full reasoning.

**Hardware:** RTX 5080 (16 GB) — inference only, ~4 GB VRAM.

Run top-to-bottom (Kernel → Restart & Run All). Writes results to `results/`.

In [ ]:
import os, random, numpy as np, torch
from dotenv import load_dotenv
load_dotenv()
assert os.environ.get("HF_TOKEN"), "Set HF_TOKEN in .env (see .env.example)"

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device:", DEVICE,
      "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")

## 1 · Build / load the three corpora

Cache-aware: each corpus is streamed once into `data/corpora/<name>/` and reused
after that, so re-running this notebook is free. `HOURS_PER_CORPUS` caps each
corpus at the same number of hours — the corpora differ in size by 8x, and an
equal cap is what stops the two big ones from setting the whole picture.

Keep this cell's `HOURS_PER_CORPUS` and `NAMES` identical to
`04_lora_finetune_3ds.ipynb`. The caches are shared, so whichever notebook runs
first fixes the splits; a mismatch silently scores different clips in the two
runs and voids the before/after diff.

In [ ]:
import corpora

# Hours streamed per corpus; None = the whole corpus.
#
# MUST match 04_lora_finetune_3ds.ipynb. The caches are shared, and whichever
# notebook builds first fixes the splits — a mismatch here means the baseline
# and the fine-tune score different clips and the before/after diff is void.
#
# 100 h x 3 = 300 h. Measured costs on this box:
#   disk      115 MB per audio-hour of 16 kHz PCM_16 wav  ->  ~35 GB
#   download  74 audio-hours per wall-hour sustained      ->  ~4 h to build
HOURS_PER_CORPUS = 100.0
# "vivoice_full" streams viVoice as its native clips and honours the cap above.
# The other entry, "vivoice", is the 8.4 h merged cache notebooks 1-2 use — it
# ignores HOURS_PER_CORPUS entirely, so swap it in only for a quick run whose
# viVoice number is directly comparable to notebook 1 (section 6).
NAMES = ["vivoice_full", "vietspeech", "vieneu"]

import shutil
free_gb = shutil.disk_usage(".").free / 1e9
need_gb = 0.115 * 3 * (HOURS_PER_CORPUS or 750)
print(f"free disk: {free_gb:.0f} GB — cached audio costs ~115 MB per audio-hour, "
      f"so this configuration needs ~{need_gb:.0f} GB\n")

# Probes access first and skips what it cannot reach, so an unapproved gated
# corpus costs a printed line rather than a crash an hour into the build.
built = corpora.prepare_all(NAMES, target_hours=HOURS_PER_CORPUS)
AVAILABLE = list(built)
print("\nusable corpora:", AVAILABLE)

In [ ]:
import pandas as pd

rows = []
for name in AVAILABLE:
    d = corpora.load_corpus(name)
    for split in corpora.SPLITS:
        rows.append({"corpus": name, "split": split, "n": len(d[split]),
                     "hours": round(sum(d[split]["duration"]) / 3600, 2),
                     # Speakers, not clips, are what a WER generalizes over: a
                     # test split resting on 2 voices measures those 2 voices.
                     "channels": len(set(d[split]["channel"]))})
comp = pd.DataFrame(rows).pivot(index="corpus", columns="split",
                                values=["n", "hours", "channels"])
print(comp.to_string())

thin = [(r["corpus"], r["split"], r["channels"]) for r in rows
        if r["split"] == "test" and r["channels"] < 5]
for name, split, n in thin:
    print(f"\nWARNING {name}: the {split} split covers only {n} channel(s). Its "
          f"WER describes those speakers, not the corpus — raise HOURS_PER_CORPUS.")

for name in AVAILABLE:
    print(f"\n{name}: {corpora.CORPORA[name].note}")
comp

## 2 · Load the model + processor

In [ ]:
from transformers import AutoProcessor, AutoModelForMultimodalLM
MODEL_ID = "Qwen/Qwen3-ASR-1.7B-hf"
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID, dtype=torch.bfloat16, attn_implementation="sdpa", device_map=DEVICE,
).eval()
print("loaded", type(model).__name__, model.dtype, model.device)

## 3 · Evaluate each corpus

`corpora.score_corpus` picks the evaluation rows, transcribes them batched and
scores with the shared `vi_norm` normalizer. Notebook 4 calls the *same*
function with the same `EVAL_LIMIT` and `SEED`, which is what makes the
before/after comparison meaningful — both runs score identical clips.

`EVAL_LIMIT` caps each corpus so a run stays in the tens of minutes. At 100 h
per corpus each test split holds thousands of clips, so all three are
subsampled; the sample is drawn by a seeded shuffle, so it is the same 500 clips
in both notebooks.

In [ ]:
import json, pathlib
import pandas as pd

EVAL_LIMIT = 500   # clips per corpus; None = the whole test split
BATCH_SIZE = 8     # lower to 4 if you hit CUDA OOM

processor.tokenizer.padding_side = "left"   # required for correct batched generation

base_metrics, base_frames = {}, {}
for name in AVAILABLE:
    n = len(corpora.eval_rows(name, EVAL_LIMIT))
    print(f"\n=== {name} · {corpora.corpus_dir(name)} [test] · n={n} ===")
    m, f = corpora.score_corpus(model, processor, name, limit=EVAL_LIMIT,
                                batch_size=BATCH_SIZE, seed=SEED)
    base_metrics[name], base_frames[name] = m, f
    print(f"   WER={m['wer']:.4f}  CER={m['cer']:.4f}  "
          f"(legacy WER={m['wer_legacy']:.4f})  n={m['n']}")

## 4 · Save results + summary

In [ ]:
pathlib.Path("results").mkdir(exist_ok=True)
for name, f in base_frames.items():
    f.to_csv(f"results/3ds_baseline_{name}_predictions.csv", index=False)
with open("results/3ds_baseline_metrics.json", "w", encoding="utf-8") as fh:
    json.dump(base_metrics, fh, ensure_ascii=False, indent=2)
print("saved -> results/3ds_baseline_metrics.json + results/3ds_baseline_<name>_predictions.csv")

summary = pd.DataFrame([{
    "corpus": name,
    "n": m["n"],
    "WER %": round(100 * m["wer"], 2),
    "CER %": round(100 * m["cer"], 2),
    "WER % (legacy)": round(100 * m["wer_legacy"], 2),
} for name, m in base_metrics.items()])
print()
print(summary.to_string(index=False))
summary

## 5 · Per-bucket breakdown

Duration matters: viVoice segments are 5–60 s while VietSpeech clips are mostly
2–6 s, so a single WER per corpus hides where the model actually struggles.
Short clips give the language-model prior less context to resolve ambiguity,
which usually shows up as a higher WER in the `0-5` bucket.

In [ ]:
from vi_norm import wer_cer

rows = []
for name, f in base_frames.items():
    for bucket, g in f.groupby("bucket"):
        m = wer_cer(g["ref"], g["hyp"])
        rows.append({"corpus": name, "bucket": bucket, "n": m["n"],
                     "WER %": round(100 * m["wer"], 2),
                     "CER %": round(100 * m["cer"], 2)})
buckets = pd.DataFrame(rows).sort_values(["corpus", "bucket"])
print(buckets.to_string(index=False))
buckets

## 6 · Sanity check against notebook 1

Only fires when `NAMES` includes the cached `"vivoice"` entry, which scores the
same 114 segments as `01_eval_baseline.ipynb` — the two numbers should then
agree to the digit, and a mismatch means something in the shared path changed
(`vi_norm`, the batching, or the cache itself).

With the default `"vivoice_full"` there is nothing to compare: that entry
streams its own clips and builds its own splits, so it shares no rows with
notebook 1. The cell prints a note and moves on.

In [ ]:
try:
    nb1 = json.load(open("results/baseline_metrics.json"))["overall"]
    here = base_metrics.get("vivoice")
    if here is None:
        print("viVoice was not evaluated in this run — nothing to compare.")
    else:
        delta = here["wer"] - nb1["wer"]
        print(f"notebook 1 viVoice WER: {nb1['wer']:.6f}  (n={nb1['n']})")
        print(f"notebook 3 viVoice WER: {here['wer']:.6f}  (n={here['n']})")
        print(f"delta: {delta:+.6f}")
        if here["n"] != nb1["n"]:
            print("NOTE: different n — EVAL_LIMIT is subsampling, so exact "
                  "agreement is not expected.")
        elif abs(delta) < 1e-9:
            print("OK — identical, as expected.")
        else:
            print("WARNING: same n but different WER. Investigate before "
                  "trusting notebook 4's before/after.")
except FileNotFoundError:
    print("No results/baseline_metrics.json — run 01_eval_baseline.ipynb "
          "to enable this check.")